# 🧠 GRPO 추론 정렬 (Group Relative Policy Optimization)

## SFT 이후, 왜 GRPO인가?

SFT(03, 04 노트북)는 **정답을 따라하는 법**을 가르칩니다.
GRPO는 **스스로 더 나은 답을 찾는 법**을 가르칩니다.

```
학습 파이프라인
 ├── 03 LoRA SFT  — 정답 따라하기 (지도 학습)
 ├── 04 OFT SFT   — 정답 따라하기 (직교 방식)
 └── 05 GRPO      — 더 나은 답 찾기 (강화 학습) ← 이 노트북
```

## GRPO란?

**GRPO** (Group Relative Policy Optimization)는 DeepSeek-R1이 사용한
강화학습 기법으로, 2025년 추론 모델 학습의 표준이 되었습니다.

### 핵심 원리
1. 프롬프트 하나에 대해 **N개의 응답**을 생성 (그룹 샘플링)
2. 각 응답을 **보상 함수**로 채점 (자동 검증, 사람 불필요)
3. 그룹 평균보다 **높은 응답은 강화**, 낮은 응답은 억제
4. 별도의 value model 없이 **그룹 자체가 baseline** 역할

```
         ┌──────────┐
         │ Prompt Q │
         └────┬─────┘
              │ 그룹 샘플링 (N=4)
    ┌─────┬──┴──┬─────┐
    ▼     ▼     ▼     ▼
  R₁=0.8 R₂=0.3 R₃=0.9 R₄=0.2   ← 보상 함수 채점
              │
         평균 = 0.55
              │
    ┌─────┬──┴──┬─────┐
    ▲     ▼     ▲     ▼
  강화   억제  강화   억제         ← 평균 대비 상대 업데이트
```

### SFT vs DPO vs GRPO 비교

| 구분 | SFT | DPO | GRPO |
|------|-----|-----|------|
| **학습 신호** | 정답 텍스트 | 선호/비선호 쌍 | 보상 점수 |
| **데이터 형태** | (prompt, answer) | (prompt, chosen, rejected) | prompt + 보상 함수 |
| **사람 개입** | 정답 작성 필요 | 선호 라벨링 필요 | **불필요** (자동 검증) |
| **강점** | 빠르고 안정적 | 정렬 품질 우수 | 추론·규칙 준수 탁월 |
| **TRL Trainer** | `SFTTrainer` | `DPOTrainer` | `GRPOTrainer` |

### PPO vs GRPO — 왜 GRPO가 효율적인가?

기존 **PPO** (Proximal Policy Optimization)는 별도의 **value model**을 유지해야 했습니다.
GRPO는 이를 **그룹 내 상대 비교**로 대체하여:
- Value model 메모리 제거 (모델 크기만큼 절약)
- 학습 안정성 향상 (baseline 추정 오류 없음)
- 구현 단순화 (보상 함수만 정의하면 됨)

### 왜 Banking 도메인에 GRPO가 적합한가?
- 은행 규칙 적용의 **정확도를 자동 검증** 가능
- "Emerald Saver의 APY 보너스는 0.75%인가?" → 정답 검증 가능 ✅
- 도구 호출 정확도, 수수료 계산 등도 자동 채점 가능 ✅
- 선호 쌍 데이터 없이 **프롬프트 + 보상 함수**만으로 학습

### 이 노트북의 설정
- **기본 모델**: 03에서 LoRA SFT로 학습한 모델 (또는 base 모델)
- **방법**: PEFT LoRA + TRL `GRPOTrainer`
- **보상 함수**: KB 문서 기반 사실 정확도 검증
- **그룹 크기**: 4 (프롬프트당 4개 응답 생성)

> ⚠️ GRPO는 SFT보다 더 많은 GPU 메모리와 시간이 필요합니다.
> 응답 생성(rollout) + 학습을 반복하기 때문입니다.

In [ ]:
"""환경 확인 — 00_preflight.ipynb 에서 이미 설치 완료."""

import os
from pathlib import Path

_nb_dir = Path.cwd()
_project_root = _nb_dir
for _p in [_nb_dir] + list(_nb_dir.parents):
    if (_p / "pyproject.toml").exists():
        _project_root = _p
        break

os.environ["RHOAI_PROJECT_ROOT"] = str(_project_root)

try:
    import rhoai_model_training_lab.config as _cfg
    _cfg.PROJECT_ROOT = _project_root
    print(f"✅ 프로젝트 루트: {_project_root}")
except ImportError:
    raise ImportError(
        "❌ 패키지 미설치 — 먼저 00_preflight.ipynb 를 실행하세요."
    )

In [ ]:
"""GRPO 설정 로드 및 SFT 모델 선택."""

import os, json
from pathlib import Path
from rhoai_model_training_lab.config import load_env, load_training_config, PROJECT_ROOT

load_env()

lora_config = load_training_config("lora")
model_id = lora_config["model"]["model_id"]

# SFT 체크포인트가 있으면 사용, 없으면 base 모델
lora_ckpt = PROJECT_ROOT / lora_config["training_args"]["output_dir"]
if (lora_ckpt / "adapter_config.json").exists():
    sft_model_path = str(lora_ckpt)
    print(f"✅ LoRA SFT 체크포인트 발견: {sft_model_path}")
    print("   GRPO는 SFT 모델 위에서 추론 정렬을 수행합니다.")
    use_sft = True
else:
    sft_model_path = model_id
    print(f"⚠️  LoRA 체크포인트 없음 — base 모델에서 시작: {model_id}")
    print("   03_lora_finetuning을 먼저 실행하면 더 좋은 결과를 얻을 수 있습니다.")
    use_sft = False

# GRPO 하이퍼파라미터
grpo_config = {
    "num_generations": 4,       # 프롬프트당 응답 수 (그룹 크기)
    "num_train_epochs": 1,      # GRPO는 보통 1 에포크
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 4,
    "learning_rate": 5e-6,      # SFT보다 훨씬 낮은 LR
    "max_completion_length": 512,
    "max_prompt_length": 512,
    "beta": 0.04,               # KL penalty
    "output_dir": "checkpoints/grpo",
    "logging_steps": 5,
    "save_steps": 50,
    "save_total_limit": 2,
}

print(f"\nGRPO Config:")
for k, v in grpo_config.items():
    print(f"  {k}: {v}")

In [ ]:
"""보상 데이터셋 구성 — 프롬프트 + 참조 답변 + KB 문서."""

import json
from datasets import Dataset

# 학습 데이터에서 프롬프트와 참조 답변 추출
train_file = str(PROJECT_ROOT / lora_config["data"]["train_file"])
with open(train_file) as f:
    raw_data = [json.loads(line) for line in f]

prompts = []
references = []
for item in raw_data:
    msgs = item["messages"]
    user_msg = next((m["content"] for m in msgs if m["role"] == "user"), None)
    asst_msg = next((m["content"] for m in msgs if m["role"] == "assistant"), None)
    if user_msg and asst_msg:
        prompts.append(user_msg)
        references.append(asst_msg)

# GRPO 데이터셋: prompt 컨럼만 필요 (참조 답변은 보상 함수에서 사용)
grpo_dataset = Dataset.from_dict({
    "prompt": prompts,
    "reference": references,
})

# 데이터 크기 제한 (GRPO는 느리므로)
max_samples = min(200, len(grpo_dataset))
grpo_dataset = grpo_dataset.shuffle(seed=42).select(range(max_samples))

print(f"📊 GRPO Dataset: {len(grpo_dataset)} prompts (from {len(prompts)} total)")
print(f"\nSample prompt:")
print(f"  Q: {grpo_dataset[0]['prompt'][:100]}...")
print(f"  Ref: {grpo_dataset[0]['reference'][:100]}...")

In [ ]:
"""보상 함수 정의 — banking 규칙 사실 정확도 검증."""

import re
from difflib import SequenceMatcher

def reward_fn(completions, prompts=None, reference=None, **kwargs):
    """Banking knowledge accuracy reward.

    GRPO는 이 함수의 반환값을 기반으로 더 나은 응답을 강화합니다.

    Scoring:
      - Similarity to reference answer (0.0 ~ 0.6)
      - Contains specific numbers/facts (0.0 ~ 0.2)
      - Response length adequacy (0.0 ~ 0.1)
      - No hallucination penalty (-0.3 if gibberish)

    Returns list of float rewards, one per completion.
    """
    rewards = []
    refs = reference if reference else [""] * len(completions)

    for completion, ref in zip(completions, refs):
        text = completion.strip()
        score = 0.0

        # 1. Semantic similarity to reference (0 ~ 0.6)
        if ref:
            sim = SequenceMatcher(None, text.lower(), ref.lower()).ratio()
            score += sim * 0.6

        # 2. Contains specific facts (numbers, percentages, dollar amounts)
        facts_in_ref = set(re.findall(r"\$[\d,.]+|\d+\.?\d*%|\d{2,}", ref))
        facts_in_text = set(re.findall(r"\$[\d,.]+|\d+\.?\d*%|\d{2,}", text))
        if facts_in_ref:
            fact_overlap = len(facts_in_ref & facts_in_text) / len(facts_in_ref)
            score += fact_overlap * 0.2

        # 3. Response length adequacy (not too short, not too long)
        words = len(text.split())
        if 10 <= words <= 200:
            score += 0.1
        elif words < 5:
            score -= 0.1

        # 4. Hallucination / gibberish penalty
        if len(text) < 3 or text.count("\n") > 20:
            score -= 0.3

        rewards.append(max(-1.0, min(1.0, score)))

    return rewards

# 보상 함수 테스트
test_ref = grpo_dataset[0]["reference"]
test_scores = reward_fn(
    completions=[test_ref, "I don't know.", ""],
    reference=[test_ref, test_ref, test_ref],
)
print("보상 함수 테스트:")
print(f"  정답과 동일:  {test_scores[0]:.3f} (high expected)")
print(f"  모호한 답변:  {test_scores[1]:.3f} (low expected)")
print(f"  빈 답변:      {test_scores[2]:.3f} (negative expected)")

In [ ]:
"""GRPO 학습 — TRL GRPOTrainer + PEFT LoRA."""

import time
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType
from trl import GRPOTrainer, GRPOConfig

if not torch.cuda.is_available():
    raise RuntimeError("GPU not detected.")

print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.1f} GB")

# ── Model ──
print("\n🔧 Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16,
    trust_remote_code=True, attn_implementation="flash_attention_2",
)

# SFT adapter가 있으면 merge
if use_sft:
    from peft import PeftModel
    model = PeftModel.from_pretrained(model, sft_model_path)
    model = model.merge_and_unload()
    print(f"  ✅ LoRA SFT adapter merged")

tokenizer = AutoTokenizer.from_pretrained(
    model_id, trust_remote_code=True, padding_side="left",
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

print(f"  {model_id} ({model.num_parameters()/1e9:.2f}B params)")

# ── GRPO용 LoRA (SFT와 별도의 새 어댑터) ──
peft_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=["q_proj", "v_proj"],
    bias="none",
    task_type=TaskType.CAUSAL_LM,
)

# ── GRPO Config ──
output_dir = str(PROJECT_ROOT / grpo_config["output_dir"])

training_args = GRPOConfig(
    output_dir=output_dir,
    num_train_epochs=grpo_config["num_train_epochs"],
    per_device_train_batch_size=grpo_config["per_device_train_batch_size"],
    gradient_accumulation_steps=grpo_config["gradient_accumulation_steps"],
    learning_rate=grpo_config["learning_rate"],
    num_generations=grpo_config["num_generations"],
    max_completion_length=grpo_config["max_completion_length"],
    max_prompt_length=grpo_config["max_prompt_length"],
    beta=grpo_config["beta"],
    bf16=True,
    logging_steps=grpo_config["logging_steps"],
    save_steps=grpo_config["save_steps"],
    save_total_limit=grpo_config["save_total_limit"],
    report_to="none",
    seed=42,
)

# ── Trainer ──
trainer = GRPOTrainer(
    model=model,
    args=training_args,
    train_dataset=grpo_dataset,
    processing_class=tokenizer,
    reward_funcs=reward_fn,
    peft_config=peft_config,
)

print("\n" + "=" * 70)
print("🚀 GRPO Training Start")
print("=" * 70)
print(f"  Prompts: {len(grpo_dataset)}")
print(f"  Generations per prompt: {grpo_config['num_generations']}")
print(f"  Epochs: {grpo_config['num_train_epochs']}")
print(f"  LR: {grpo_config['learning_rate']} | Beta (KL): {grpo_config['beta']}")
print(f"  LoRA r=8 (GRPO adapter)")
print()

start_time = time.time()
training_result = trainer.train()
wall_time = time.time() - start_time

print(f"\n✅ GRPO Training Complete!")
print(f"  Wall time: {wall_time/60:.1f} min")
print(f"  Peak VRAM: {torch.cuda.max_memory_allocated()/(1024**3):.1f} GB")

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"  Model saved: {output_dir}")

In [ ]:
"""GRPO Training analysis — reward curves, policy loss, KL divergence."""

log_history = trainer.state.log_history
print(f"Log entries: {len(log_history)}")

# GRPO logs: reward/mean, reward/std, kl, loss
steps = [e["step"] for e in log_history if "loss" in e]
losses = [e["loss"] for e in log_history if "loss" in e]
reward_steps = [e["step"] for e in log_history if "reward" in e]
rewards_mean = [e["reward"] for e in log_history if "reward" in e]
kl_steps = [e["step"] for e in log_history if "kl" in e]
kl_values = [e["kl"] for e in log_history if "kl" in e]

print(f"  Loss: {len(losses)} | Reward: {len(rewards_mean)} | KL: {len(kl_values)}")

if losses or rewards_mean:
    try:
        import matplotlib.pyplot as plt

        n_plots = bool(losses) + bool(rewards_mean) + bool(kl_values)
        fig, axes = plt.subplots(1, max(n_plots, 1), figsize=(5 * max(n_plots, 1), 4.5))
        if n_plots == 1:
            axes = [axes]
        ax_idx = 0

        if losses:
            ax = axes[ax_idx]; ax_idx += 1
            ax.plot(steps, losses, color="#4C72B0", lw=1.5)
            ax.set_xlabel("Step"); ax.set_ylabel("Loss")
            ax.set_title("GRPO Policy Loss"); ax.grid(True, alpha=0.3)

        if rewards_mean:
            ax = axes[ax_idx]; ax_idx += 1
            ax.plot(reward_steps, rewards_mean, color="#55A868", lw=1.5, marker="o", ms=3)
            ax.set_xlabel("Step"); ax.set_ylabel("Mean Reward")
            ax.set_title("Reward Trend"); ax.grid(True, alpha=0.3)
            ax.axhline(y=rewards_mean[0], color="gray", ls="--", alpha=0.4, label="Initial")
            ax.legend(fontsize=8)

        if kl_values:
            ax = axes[ax_idx]; ax_idx += 1
            ax.plot(kl_steps, kl_values, color="#C44E52", lw=1.5)
            ax.set_xlabel("Step"); ax.set_ylabel("KL Divergence")
            ax.set_title("KL from Reference"); ax.grid(True, alpha=0.3)

        fig.suptitle("GRPO Training Analysis", fontsize=13, fontweight="bold", y=1.02)
        plt.tight_layout()
        plt.show()

    except ImportError:
        pass

    print("\n" + "=" * 60)
    print("GRPO Metrics Summary")
    print("=" * 60)
    if losses:
        print(f"  Initial loss:  {losses[0]:.4f}")
        print(f"  Final loss:    {losses[-1]:.4f}")
    if rewards_mean:
        print(f"  Initial reward: {rewards_mean[0]:.4f}")
        print(f"  Final reward:   {rewards_mean[-1]:.4f}")
        print(f"  Max reward:     {max(rewards_mean):.4f}")
        improvement = rewards_mean[-1] - rewards_mean[0]
        print(f"  Improvement:    {improvement:+.4f}")
    if kl_values:
        print(f"  Final KL:       {kl_values[-1]:.4f}")
    print(f"\n  Wall time:     {wall_time/60:.1f} min")
    print(f"  Peak VRAM:     {torch.cuda.max_memory_allocated()/(1024**3):.1f} GB")
else:
    print("⚠️  No training metrics found.")

In [ ]:
"""학습 결과 로컬 저장."""

result_summary = {
    "method": "grpo",
    "model_id": model_id,
    "sft_base": sft_model_path if use_sft else None,
    "num_prompts": len(grpo_dataset),
    "num_generations": grpo_config["num_generations"],
    "learning_rate": grpo_config["learning_rate"],
    "beta": grpo_config["beta"],
    "lora_r": 8,
    "wall_time_seconds": wall_time,
    "peak_vram_gb": torch.cuda.max_memory_allocated() / (1024**3),
    "gpu_name": torch.cuda.get_device_name(0),
    "final_loss": losses[-1] if losses else None,
    "final_reward": rewards_mean[-1] if rewards_mean else None,
    "max_reward": max(rewards_mean) if rewards_mean else None,
    "final_kl": kl_values[-1] if kl_values else None,
}

result_path = Path(output_dir) / "training_result.json"
result_path.parent.mkdir(parents=True, exist_ok=True)
with open(result_path, "w") as f:
    json.dump(result_summary, f, indent=2, ensure_ascii=False)
print(f"📄 학습 결과 저장: {result_path}")

In [ ]:
"""GRPO 전후 응답 비교 — 질적 평가."""

print("=" * 70)
print("📝 GRPO 학습 후 응답 확인")
print("=" * 70)

test_prompts = [
    "What is the monthly maintenance fee for the Dark Green Account?",
    "How does the APY bonus work for the Emerald Saver Account?",
    "What are the foreign ATM withdrawal fees for the Dark Green Account?",
]

model.eval()
for i, prompt in enumerate(test_prompts):
    print(f"\n--- Question {i+1} ---")
    print(f"Q: {prompt}")

    inputs = tokenizer(
        tokenizer.apply_chat_template(
            [{"role": "user", "content": prompt}],
            tokenize=False, add_generation_prompt=True
        ),
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=200,
            temperature=0.7, do_sample=True,
        )

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True
    )
    print(f"A: {response[:300]}")

print()
print("=" * 70)
print("다음 단계:")
print("  📓 06_export_and_deploy.ipynb — 모델 내보내기 및 배포")
print("  📓 08_compare_results.ipynb — SFT vs GRPO 비교 평가")